In [ ]:
# Useful for: I/O-bound work like HTTP requests, database queries, and file operations, where your code waits for external systems.

# Concepts
# Coroutines: Functions defined with async def instead of def. They can pause and resume execution, making them perfect for operations that involve waiting.
# await: This keyword tells Python, "pause this coroutine until this operation completes, but let other code run in the meantime."
# Event loop: The engine that manages all your coroutines, deciding which one to run and when to switch between them.
# Tasks: Coroutines wrapped for concurrent execution. You create them with asyncio.create_task() to run multiple operations at once.

In [20]:
import asyncio
import aiohttp
import time

In [ ]:
# Pauses function execution for 2 seconds (but would allow other operations to continue)
async def greet_after_delay():
    print("Starting...")
    await asyncio.sleep(2)  # Pauses, but doesn't block
    print("Hello!")

# asyncio.run(greet_after_delay())
await greet_after_delay()
# asyncio.run create event loop and by executing function; await executes function within event loop
# Use await instead of asyncio.run in Jupyter - event loop already running

Starting...
Hello!


In [12]:
# Chained functions
async def get_message():
    await asyncio.sleep(1)
    return "Hello!"

async def main():
    message = await get_message()
    print(message)

await main()

Hello!


In [ ]:
# Use asyncio.gather() to run multiple functions concurrently
async def greet_after_delay(name):
    print(f"Starting {name}...")
    await asyncio.sleep(2)
    print(f"Hello, {name}!")

async def main():
    start = time.perf_counter()
    
    await asyncio.gather(
        greet_after_delay("Alice"),
        greet_after_delay("Bob"),
        greet_after_delay("Charlie"),
    )
    
    elapsed = time.perf_counter() - start
    print(f"Total time: {elapsed:.2f} seconds")

await main()

Starting Alice...
Starting Bob...
Starting Charlie...
Hello, Alice!
Hello, Bob!
Hello, Charlie!
Total time: 2.01 seconds


In [19]:
# asyncio.gather() returns compiled values from functions
# (in order listed, not in order completed)
async def fetch_number(n):
    await asyncio.sleep(n)
    return n * 10

async def main():
    results = await asyncio.gather(
        fetch_number(1),
        fetch_number(2),
        fetch_number(3),
    )
    print(results)

await main()

[10, 20, 30]


In [ ]:
# Use aiohttp.ClientSession().get(<URL>) to concurrently fetch HTML content from a URL
async def fetch(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            return await response.text()

async def main():
    html = await fetch("https://example.com")
    print(f"Fetched {len(html)} characters")

await main()

Fetched 528 characters


In [24]:
# Define aiohttp.ClientSession() object once for multiple requests to save time by reusing TCP connection 
async def fetch_good(session, url):
    async with session.get(url) as response:
        return await response.text()

async def main():
    urls = ["https://example.com"] * 10
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[fetch_good(session, url) for url in urls])
    return results

results = await main()
[len(i) for i in results]

[528, 528, 528, 528, 528, 528, 528, 528, 528, 528]

In [ ]:
# API example

HN_API = "https://hacker-news.firebaseio.com/v0"

async def main():
    async with aiohttp.ClientSession() as session:
        # Get top story IDs
        async with session.get(f"{HN_API}/topstories.json") as response:
            story_ids = await response.json()
        
        print(f"Found {len(story_ids)} stories")
        print(f"First 5 IDs: {story_ids[:5]}")
        
        # Fetch first story details
        first_id = story_ids[0]
        async with session.get(f"{HN_API}/item/{first_id}.json") as response:
            story = await response.json()
        
        print(f"\nStory structure:")
        for key, value in story.items():
            print(f"  {key}: {repr(value)[:50]}")

await main()

Found 500 stories
First 5 IDs: [47865868, 47863217, 47861270, 47862497, 47862386]

Story structure:
  by: 'Kaibeezy'
  descendants: 68
  id: 47865868
  kids: [47866242, 47866318, 47866591, 47866134, 47866220,
  score: 235
  time: 1776875365
  title: 'Alberta startup sells no-tech tractors for half p
  type: 'story'
  url: 'https://wheelfront.com/this-alberta-startup-sells
